In [ ]:
#install required packages and import required functions for post-processing
import importlib
import subprocess
import sys

required_packages = {
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "os": None,  # Built-in, no installation needed
    "scipy": "scipy",
    "numpy": "numpy",
}

for module_name, package_name in required_packages.items():
    if package_name is None:
        continue  # Skip built-in modules
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Now import all modules (safe to do after installation)
import pandas as pd
from sklearn.cluster import DBSCAN
import os
from scipy.optimize import minimize
import numpy as np

def split_point_cloud_by_annotation(file_path):
    # Read the data from the text file
    df = pd.read_csv(file_path, sep='\s+', header=None, names=['x', 'y', 'z', 'annotation'])
    # Separate the data based on annotation
    zMin = df['z'].min()
    stem = df[df['annotation'] == 1]
    dead = df[df['annotation'] == 2]
    living = df[df['annotation'] == 3]
    return stem, dead, living, zMin

def cluster_point_cloud_with_dbscan(df, eps=0.15, min_samples=5):
    # Extract the coordinates
    X = df[['x', 'y', 'z']].values
    # Apply DBSCAN
    db = DBSCAN(eps=eps, min_samples=min_samples)
    df['cluster'] = db.fit_predict(X)
    df = df[df['cluster'] != -1]
    return df

def remove_small_clusters(df, min_size):
    # Calculate the size of each cluster
    cluster_sizes = df['cluster'].value_counts()
    # Get clusters that meet the minimum size requirement
    large_clusters = cluster_sizes[cluster_sizes >= min_size].index
    # Filter the DataFrame to keep only large clusters
    filtered_df = df[df['cluster'].isin(large_clusters)]
    return filtered_df

def filter_clusters_by_centroid_distance(df, distance_threshold):
    # Calculate centroids for each cluster
    centroids = df.groupby('cluster')[['x', 'y', 'z']].mean()
    # Identify the largest cluster by point count
    largest_cluster_label = df['cluster'].value_counts().idxmax()
    # Get the centroid of the largest cluster
    largest_cluster_centroid = centroids.loc[largest_cluster_label]
    # Function to calculate Euclidean distance between centroids
    def euclidean_distance(centroid1, centroid2):
        return np.linalg.norm(centroid1 - centroid2)
    # Calculate distances between the largest cluster centroid and others
    distances = centroids.apply(lambda centroid: euclidean_distance(largest_cluster_centroid, centroid), axis=1)
    # Filter clusters within the given distance threshold
    valid_clusters = distances[distances <= distance_threshold].index
    # Filter the original DataFrame
    filtered_df = df[df['cluster'].isin(valid_clusters)]
    return filtered_df

def filter_clusters_by_distance(df, range_threshold):
    # Identify the largest cluster by point count
    largest_cluster_label = df['cluster'].value_counts().idxmax()
    # Extract the largest cluster's data
    largest_cluster_data = df[df['cluster'] == largest_cluster_label]
    # Find the lowest point (minimum z) in the largest cluster
    largest_cluster_lowest_z = largest_cluster_data['z'].min()
    # Function to get the lowest point in each cluster
    def get_cluster_lowest_z(cluster_label):
        cluster_data = df[df['cluster'] == cluster_label]
        return cluster_data['z'].min()
    # Get the unique clusters
    unique_clusters = df['cluster'].unique()
    # Filter clusters whose lowest point is within the range threshold from the largest cluster's lowest point
    valid_clusters = [
        cluster_label for cluster_label in unique_clusters
        if abs(get_cluster_lowest_z(cluster_label) - largest_cluster_lowest_z) <= range_threshold
    ]
    # Filter the DataFrame to only include valid clusters
    filtered_df = df[df['cluster'].isin(valid_clusters)]
    return filtered_df

def fit_circle(x, y):
    def objective(params):
        """
        Objective function to minimize.
        """
        xc, yc, r = params
        return np.sum((np.sqrt((x - xc)**2 + (y - yc)**2) - r)**2)
    
    # Initial guess: center (0,0) and radius = 1
    #print(x)
    if len(x) == 0 or len(y) == 0:
        xc, yc, r = 0,0,0
    else:
        x_middle = (x.max()+x.min())/2
        y_middle = (y.max()+y.min())/2
        initial_guess = (x_middle, y_middle, 0.2)
        # Perform optimization
        result = minimize(objective, initial_guess, method='L-BFGS-B', bounds=[(-np.inf, np.inf), (-np.inf, np.inf), (0, np.inf)])
        xc, yc, r = result.x
    return xc, yc, r

def get_circle_parameters(df):
    #df = df[(df['z'] >= 1) & (df['z'] <= 1.5)] #filter points between 1-1.5 almost breash height
    # Extract x and y coordinates
    x = df['x'].values
    y = df['y'].values
    # Fit a circle to the points
    xc, yc, r = fit_circle(x, y)
    #print("Middle x,y and radius:")
    #print(xc, yc, r)
    return xc, yc, r

def get_stem_model(df,freg):
    zMax = df['z'].max()
    zMin = df['z'].min()
    stem = pd.DataFrame(columns=['x', 'y', 'radius','h'])
    times = (zMax-zMin)/freg #how many circles
    h = zMin
    for i in range(int(times)):
        slice = df[(df['z'] >= h) & (df['z'] <= h+freg)] 
        if len(slice) != 0:
            center_x, center_y, radius = get_circle_parameters(slice)
        else:
            break
        stem.loc[len(stem)] = [center_x, center_y, radius, (h)]
        h = h+freg
    # Step 1: Sort the DataFrame by 'h'
    stem_sorted = stem.sort_values(by='h').reset_index(drop=True)
    # Step 2: Identify rows to keep
    rows_to_keep = []
    # Initialize the last_radius variable to None
    last_radius = None
    for i in range(len(stem_sorted)):
        current_radius = stem_sorted.loc[i, 'radius']
        # If last_radius is None, it means we're on the first row
        if last_radius is None:
            rows_to_keep.append(i)
            last_radius = current_radius
        else:
            # Check if the increase in radius is more than 0.02
            if (current_radius - last_radius) <= 0.02:
                rows_to_keep.append(i)
                last_radius = current_radius
    # Step 3: Filter the DataFrame to keep only the required rows
    stem_filtered = stem_sorted.loc[rows_to_keep].reset_index(drop=True)

    return stem_filtered

def filter_points_in_circle(df, center, radius):
    # Extract x, y coordinates from the DataFrame
    x = df['x']
    y = df['y']
    # Extract center coordinates
    center_x, center_y = center
    # Compute squared distance from the center
    squared_distance = (x - center_x)**2 + (y - center_y)**2
    # Apply radius filter
    squared_radius = radius**2
    filtered_df = df[squared_distance > squared_radius]
    #if(len(df)-len(filtered_df))>0:
        #print("Filtered points by radius:"+str(len(df)-len(filtered_df)))
    return filtered_df

def filter_with_stem(df,stemModel): #filter points in the stem model circles
    buffer = 0.03 #extra radius for filtering cm

    #delete point which are lower to stem min point
    df = df[df['z'] >= stemModel['h'].min()]
        
    for i in range(len(stemModel)-1):
        h = stemModel['h'][i]
        x = stemModel['x'][i]
        y = stemModel['y'][i]
        radius = stemModel['radius'][i]+buffer
        #print(h,x,y,radius)
        slice = df[(df['z'] >= h) & (df['z'] <= h+stemModel['h'][i+1])]
        #if len(slice) > 0:
            #print("Slice time!: "+str(len(slice)))
        df = df[(df['z'] <= h) | (df['z'] >= (h+stemModel['h'][i+1]))]
        #print("Rest of df: "+str(len(df)))
        slice_filtered = filter_points_in_circle(slice, (x,y) , radius)
        df = pd.concat([df, slice_filtered], axis=0)
    
    return df

def get_dead_canopy(df, stem_df):
    #print("asds")
    # Step 1: Find the minimum z value for each cluster
    min_z_per_cluster = df.groupby('cluster')['z'].min()
    # Step 2: Identify the cluster with the lowest minimum z value
    lowest_cluster = min_z_per_cluster.idxmin()
    # Step 3: Filter the DataFrame to get rows corresponding to the lowest cluster
    lowest_cluster_df = df[df['cluster'] == lowest_cluster]
    # Step 1: Extract the lowest z value from lowest_cluster_df
    lowest_z = lowest_cluster_df['z'].min()
    # Step 2: Calculate the absolute difference in height (h) from the lowest z
    stem_df['height_difference'] = abs(stem_df['h'] - lowest_z)
    # Step 3: Find the row with the minimum difference
    closest_stem_idx = stem_df['height_difference'].idxmin()
    # Step 4: Extract the corresponding row
    closest_stem = stem_df.loc[closest_stem_idx]
    # Step 1: Extract x, y coordinates from closest_stem
    stem_x = closest_stem['x']
    stem_y = closest_stem['y']
    h = closest_stem['h']
    radius = closest_stem['radius']
    # Compute horizontal (xy) distance to stem center
    d_xy = np.sqrt((lowest_cluster_df['x'] - stem_x)**2 + 
                (lowest_cluster_df['y'] - stem_y)**2)
    # Compute absolute vertical distance (difference in z-coordinates)
    d_z = np.abs(lowest_cluster_df['z'] - h)  # h is stem height at that (x, y)
    # Compute 3D distance to stem surface
    lowest_cluster_df['distance_to_stem'] = np.sqrt(
        (np.maximum(d_xy - radius, 0))**2 + d_z**2
    )
    # Sort by distance (ascending), then by z (ascending)
    lowest_cluster_df = lowest_cluster_df.sort_values(by=['distance_to_stem', 'z'], ascending=[True, True])
    # Select the best point
    closest_point = lowest_cluster_df.iloc[0]
    # Check if closest_point is a DataFrame (multiple rows)
    if isinstance(closest_point, pd.DataFrame):
        closest_point = closest_point.iloc[0]  # Take the first row if there are multiple
    return closest_point['z']


In [6]:
#Post-process segmented Spruce or Pine point clouds from which ground, foliage and points of other trees are filtered. Get predicted boundary heights of living and dead crown.
folder = os.getcwd() #set working directory (default ipynb location)
os.chdir(folder)
%cd $folder
print("Current folder:"+os.getcwd())
cloud_path = folder+'/predict/' 
all_files = os.listdir(cloud_path)
destination_path = folder+'/postProcessed/' #Destination path of post-processed clouds.
pine = False #True if Scots pine. False if Norway spruce. Demo point clouds are Norway spruces.
external = True

if pine: #Processing parameters of pine
    min_sizeDead = 20 #min cluster size
    min_sizeLiving = 300 #min cluster size
    epsLiving = 0.15 #epsilon of dbscan in living crown clustering
    epsDead = 0.025 #epsilon of dbscan in dead crown clustering
    livingRange = 2.5 #range in meters where living crown cluster are filtered if cluster distance exceed from main living clusters bottom height
else: #Processing parameters of spruce
    min_sizeDead = 20 
    min_sizeLiving = 300 
    epsLiving = 0.15
    epsDead = 0.025
    livingRange = 1

#create results dataframe
columns = ['Tree', 'lbpred','lbpredSimple' ,'dbpred','dbpredSimple']
predict = pd.DataFrame(columns=columns)

#loop trough folder of point cloud folder
for x in all_files:
    file = x
    if ".txt" not in file: #break if not .txt file
        print("not txt!")
        break
    stem, dead, living , zMin = split_point_cloud_by_annotation(cloud_path+file) #import point cloud and split it by segmentation
    print(file)

    # DBSCAN clustering of stem points
    clustered_stem = cluster_point_cloud_with_dbscan(stem,0.05) #cluster stem points
    filtered_stem = remove_small_clusters(clustered_stem, min_sizeLiving) #filter stem points by removing small clusters
    stem = get_stem_model(filtered_stem,0.2) #get stem model using filtered stem poinst

    # DBSCAN clustering of living and dead crown points.
    clustered_living = cluster_point_cloud_with_dbscan(living,epsLiving)
    clustered_dead = cluster_point_cloud_with_dbscan(dead,epsDead)

    # Filter living and dead crown points by removing points inside of stem model slices.
    clustered_dead = filter_with_stem(clustered_dead,stem)
    clustered_living = filter_with_stem(clustered_living,stem)
    
    # Filter small clusters of living and dead crown points
    filtered_living = filter_clusters_by_distance(clustered_living,livingRange)
    filtered_dead = remove_small_clusters(clustered_dead, min_sizeDead) 
    
    # The estimation of boundaries of the living and dead canopies by identifying the lowest cluster of the respective point class.
    # The nearest slice of the stem model to the lowest point of the cluster was selected as a reference for measuring the distance between the cluster points and the stem. 
    # Finally, the boundaries of the living and dead crowns were determined by selecting the points closest to the stem surface.
    dead_point = get_dead_canopy(filtered_dead,stem)
    living_point = get_dead_canopy(filtered_living,stem) 

    #get simple crown boundaries by exrtacting lowest points of respective classe.
    dead_lowest = dead['z'].min()
    living_lowest = living['z'].min()

    #Add boundaries to dataframe
    ogTree = file.replace(".txt", "")
    new_row = pd.DataFrame({'Tree': [file.replace(".txt", "")], 'lbpred': 
                            [living_point],'lbpredSimple':
                            [living_lowest],'dbpred': [dead_point],'dbpredSimple':
                            [dead_lowest],})
    combined_filtered_df = pd.concat([filtered_living, filtered_dead, filtered_stem], ignore_index=True)
    combined_filtered_df.to_csv(destination_path+file, sep=' ', index=False, header=False)
    predict = predict.dropna(axis=1, how='all')
    predict = pd.concat([predict, new_row], ignore_index=True)

print('Boundaries are bounded!')

C:\Users\mikapehk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


c:\LocalData\mikapehk\DeadLivingCanopy\Living and Dead canopy boundaries
Current folder:c:\LocalData\mikapehk\DeadLivingCanopy\Living and Dead canopy boundaries
119.txt


C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(
C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(


119A1.txt


C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(
C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(


...

626.txt


C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(
C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(


626A1.txt


C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(
C:\Users\mikapehk\AppData\Local\Temp\ipykernel_2368\1422405289.py:266: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lowest_cluster_df['distance_to_stem'] = np.sqrt(


not txt!
Boundaries are bounded!
